# Use Case 1 — Industrial Turbine TKG: Dataset Generation & Exploration

**Domain:** Industrial predictive maintenance — synthetic turbine sensor data.

**Dataset:** 5 sensors over 30 days @ 10s intervals (~1.3M observations). Three injected anomaly types:
- **Spike** on VIB_001 (day 7, 2h) — sudden 5x magnitude
- **Gradual degradation** on TEMP_001 (days 15-25) — +20°C linear drift
- **Cyclic anomaly** on PRES_001 (day 20, 4h) — sinusoidal oscillation

**TKG Schema:**
```
(Component)-[:HAS_SENSOR]->(Sensor)-[:MADE_OBSERVATION]->(Observation)
(Observation)-[:DETECTED_ANOMALY]->(AnomalyEvent)
```

**Notebooks:**
- **01 (this):** Generate dataset + explore sensor patterns
- **02:** IsolationForest baseline vs TGN comparison
- **03:** Neo4j import + temporal queries


## 0. Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = Path('../../data/raw/synthetic_turbine.csv')
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
print('Setup OK')


## 1. Generate Dataset


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('gen', '../../data/UseCase1/generate_synthetic.py')
gen = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gen)
df_raw = gen.generate(str(DATA_PATH))


## 2. Load & Explore


In [ ]:
df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f'Shape: {df.shape}')
print(f'Period: {df["timestamp"].min().date()} -> {df["timestamp"].max().date()}')
print(f'Sensors: {df["sensor_id"].unique().tolist()}')
print(f'Anomaly rate: {df["is_anomaly"].mean()*100:.1f}%')
print()
print(df.groupby(['sensor_id', 'anomaly_type']).size().rename('count').to_string())


In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
sensors = ['TEMP_001', 'PRES_001', 'VIB_001', 'FLOW_001', 'RPM_001']
atype_color = {'none': 'steelblue', 'spike': 'crimson',
               'gradual_degradation': 'orange', 'cyclic_anomaly': 'purple'}

for ax, sid in zip(axes, sensors):
    sub = df[df['sensor_id'] == sid]
    for atype, grp in sub.groupby('anomaly_type'):
        ax.plot(grp['timestamp'], grp['value'], '.', ms=0.4,
                color=atype_color.get(atype, 'gray'), alpha=0.5)
    meta = sub.iloc[0]
    ax.set_ylabel(f"{meta['sensor_name']}\n({meta['unit']})", fontsize=8)
    ax.set_title(sid, fontsize=9)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[-1].set_xlabel('Date')
plt.suptitle('Industrial Turbine — 5 Sensors (30 days)', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()


## 3. Anomaly Zoom


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Spike VIB_001 day 7
vib = df[df['sensor_id'] == 'VIB_001']
d7 = vib[vib['timestamp'].dt.day == 8]
axes[0].plot(d7['timestamp'], d7['value'], '.', ms=0.8, color='steelblue', alpha=0.4)
sp = d7[d7['anomaly_type'] == 'spike']
axes[0].plot(sp['timestamp'], sp['value'], '.', ms=2, color='crimson', label='Spike')
axes[0].set_title('VIB_001 — Spike (Day 7)')
axes[0].set_ylabel('mm/s')
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', rotation=30)

# Gradual degradation TEMP_001
temp = df[df['sensor_id'] == 'TEMP_001']
axes[1].plot(temp['timestamp'], temp['value'], '.', ms=0.2, alpha=0.3, color='steelblue')
axes[1].axvspan(datetime(2024,1,16), datetime(2024,1,26), alpha=0.15, color='orange', label='Degradation')
axes[1].set_title('TEMP_001 — Gradual Degradation (Days 15-25)')
axes[1].set_ylabel('°C')
axes[1].legend(fontsize=8)

# Cyclic PRES_001 day 20
pres = df[df['sensor_id'] == 'PRES_001']
d20 = pres[pres['timestamp'].dt.day == 21]
axes[2].plot(d20['timestamp'], d20['value'], '.', ms=0.8, color='steelblue', alpha=0.4)
cy = d20[d20['anomaly_type'] == 'cyclic_anomaly']
axes[2].plot(cy['timestamp'], cy['value'], '.', ms=2, color='purple', label='Cyclic')
axes[2].set_title('PRES_001 — Cyclic Anomaly (Day 20)')
axes[2].set_ylabel('bar')
axes[2].legend(fontsize=8)

for ax in axes:
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


## 4. TKG Entity Summary


In [ ]:
components = df['component_id'].unique()
sensors    = df['sensor_id'].unique()
obs_total  = len(df)
anom_total = df['is_anomaly'].sum()

print('TKG Entity Summary')
print('-' * 40)
print(f'  Components   : {len(components)} -> {list(components)}')
print(f'  Sensors      : {len(sensors)}')
print(f'  Observations : {obs_total:,}')
print(f'  AnomalyEvents: {anom_total:,} ({anom_total/obs_total*100:.1f}%)')
print()
print('Relations')
print(f'  HAS_SENSOR       : {len(sensors)}')
print(f'  MADE_OBSERVATION : {obs_total:,}')
print(f'  DETECTED_ANOMALY : {anom_total:,}')


## Summary

| Entity | Count |
|--------|-------|
| Components | 1 (TURBINE_001) |
| Sensors | 5 |
| Observations | ~1.3M |
| AnomalyEvents | ~90k (6.8%) |

Three anomaly types injected with ground truth labels — ideal for supervised and unsupervised detection.

**Next:** `02_anomaly_detection.ipynb` — IsolationForest baseline vs TGN
